# Exercise 4: Short-Time Fourier Transform

This exercise develops understanding of the Short-Time Fourier Transform (STFT) and its applications in audio analysis. It has four parts:
1. Extract the main lobe of the spectrum of a window
2. Measure reconstruction noise using the STFT model
3. Compute band-wise energy envelopes of an audio signal
4. Compute an onset detection function

### Relevant concepts

**Main lobe of a window spectrum:** The width of the main lobe of a window's magnitude spectrum determines the frequency resolution of the analysis. Wider main lobes mean poorer frequency resolution but lower side lobes (better dynamic range). Importantly, changing the window length $M$ does **not** change the main-lobe width *in samples* — it only changes the frequency resolution in Hz. Zero-padding by a factor $P$ scales the main-lobe width by $P$ samples but not in normalized frequency.

**Fast Fourier Transform (FFT):** The FFT exploits the symmetry of the DFT matrix to reduce computation from $O(N^2)$ to $O(N \log N)$. It is most efficient when $N$ is a power of 2, which is the standard choice in audio processing.

**Energy of a signal:** The energy of a discrete-time signal $x[n]$ of length $N$ is:

$$E = \sum_{n=0}^{N-1} |x[n]|^2$$

**Energy in a frequency band:** Given a DFT spectrum $X[k]$ (in linear scale), the energy in the band spanning bins $k_1$ to $k_2$ is:

$$E = \sum_{k=k_1}^{k_2} |X[k]|^2$$

The `stftAnal()` function returns magnitudes in dB — convert to linear scale before summing energies. The result can then be expressed in dB as $E_{\mathrm{dB}} = 10\log_{10}(E)$.

**Signal-to-noise ratio (SNR):** SNR quantifies the amount of distortion introduced by analysis and synthesis. In the context of the STFT model:

$$\mathrm{SNR} = 10\log_{10}\!\left(\frac{E_{\mathrm{signal}}}{E_{\mathrm{noise}}}\right)$$

where $E_{\mathrm{noise}}$ is the energy of the difference between the original and reconstructed signal.

**Onset detection function (ODF):** An ODF assigns one value per audio frame and has high values at the onset of acoustic events. A simple energy-based ODF measures the increase in band energy between consecutive frames:

$$O(l) = E(l) - E(l-1), \quad l \geq 1$$

with initial condition $O(0) = 0$. To detect only onsets (not offsets), apply **half-wave rectification**:

$$\bar{O}(l) = \max(O(l),\; 0)$$


## Part 1 – Extracting the main lobe of the spectrum of a window

The function `extract_main_lobe()` should extract the main lobe of the magnitude spectrum of a window and return it in dB.

**Steps:**
1. Generate a window of type `window` and length `M` using `get_window()`.
2. Compute its `N`-point FFT with zero-phase windowing. Use `N = 8*M` (need not be a power of 2 for this part).
3. Compute the magnitude spectrum in dB: $m_W = 20\log_{10}(|W| + \varepsilon)$ where `eps = np.finfo(float).eps`.
4. Use `fftshift()` to centre the spectrum around DC.
5. Find the two local minima flanking the main lobe (the first minima on either side of the peak). Return the samples from the left minima to the right minima **inclusive**.

**Inputs:**
- `window` (str): window type — `'boxcar'`, `'hamming'`, or `'blackmanharris'`
- `M` (int): window length
- `N` (int): FFT size

**Output:** np.array containing the main-lobe samples in dB (including both boundary minima).

> Tip: scan outward from the peak index and stop as soon as the spectrum starts rising again to locate each minima.


In [5]:
import numpy as np
from scipy.signal import get_window
from scipy.fftpack import fft, fftshift
import matplotlib.pyplot as plt
eps = np.finfo(float).eps
from smstools.models import stft
from smstools.models import utilFunctions as UF

In [6]:
# E4 - 1.1: Complete function extract_main_lob()

def extract_main_lobe(window, M, N):
    """Extract the main lobe of the magnitude spectrum of a window, given a window type and its length.

    Args:
        window (str): Window type to be used (either rectangular ('boxcar'), 'hamming' or 'blackmanharris')
        M (int): length of the window to be used
        N (int): size of FFT

    Results:
        np.array: an array containing the main lobe of the magnitude spectrum of the window in decibels (dB).
    """

    w = get_window(window, M)         # get the window

    ### Your code here


Test cases for `extract_main_lobe()`:

**Test case 1:** `window = 'blackmanharris'`, `M = 100`, `N = 800` → output has **65 samples**.

**Test case 2:** `window = 'boxcar'`, `M = 120`, `N = 960` → output has **17 samples**.

**Test case 3:** `window = 'hamming'`, `M = 256`, `N = 2048` → output has **33 samples**.

Plot the full magnitude spectrum of each window (x-axis in normalized samples or bins) and mark the two boundary minima of the main lobe. Compute the **normalized main-lobe width** by dividing the sample count by the zero-padding factor ($N/M$) and compare with the theoretical values from the lectures.


In [ ]:
# E4 - 1.2: Call extract_main_lobe() for the 3 test cases, plot the magnitude spectrum of each window
#            with the main-lobe boundaries marked. Compute and report the normalized main-lobe width.

### Your code here


**E4 - 1.3: Conceptual questions (answer here)**

1. From your plots, report the normalized main-lobe width (in samples) for each of the three window types. How do they compare to the theoretical values (`boxcar` = 2, `hamming` = 4, `blackmanharris` = 8 bins in normalized frequency)?
2. The description says that changing `M` does not affect the main-lobe width in samples. Verify this: run `extract_main_lobe('hamming', M=128, N=1024)` and `extract_main_lobe('hamming', M=256, N=2048)`. Are the output lengths the same? Explain why.
3. What is the tradeoff between main-lobe width and side-lobe attenuation? Given this tradeoff, which window would you choose for analysing a signal with two closely spaced frequency components, and which for a signal where one component is much weaker than another?


## Part 2 – Measuring reconstruction noise using the STFT model

The function `compute_snr()` should quantify the distortion introduced by STFT analysis followed by synthesis, using the signal-to-noise ratio (SNR) in dB.

**Steps:**
1. Load the audio file using `UF.wavread()`.
2. Run STFT analysis and synthesis via `stft.stftAnal()` and `stft.stftSynth()`.
3. Trim the output to the same length as the input (synthesis may be slightly longer).
4. Compute the noise signal as `noise = input_signal - output_signal`.
5. Calculate two SNR values:
   - **SNR1**: over the full signal length
   - **SNR2**: after discarding the first and last `M` samples (reduces boundary effects)

**Inputs:**
- `input_file` (str): path to a `.wav` file
- `window` (str): analysis window type
- `M` (int): window length (odd)
- `N` (int): FFT size (power of 2, $N > M$)
- `H` (int): hop size

**Output:** tuple `(SNR1, SNR2)` — both values in dB.


In [8]:
# E4 - 2.1: Complete function compute_snr()

def compute_snr(input_file, window, M, N, H):
    """Measure the amount of distortion introduced during the analysis and synthesis of a signal using the STFT model.

    Args:
        input_file (str): wav file name including the path
        window (str): analysis window type (rectangular, triangular, hanning, hamming, blackman, or blackmanharris)
        M (int): analysis window length (odd positive integer)
        N (int): fft size (power of two, > M)
        H (int): hop size for the stft computation

    Result:
        tuple with the signal to noise ratio over the whole sound and of the sound without the begining and end.
    """
    ### your code here


Test cases for `compute_snr()`:

**Test case 1:** `piano.wav`, `window = 'blackman'`, `M = 513`, `N = 2048`, `H = 128`
→ expected: `(SNR1 ≈ 67.6 dB, SNR2 ≈ 86.4 dB)`

**Test case 2:** `sax-phrase-short.wav`, `window = 'hamming'`, `M = 512`, `N = 1024`, `H = 64`
→ expected: `(SNR1 ≈ 89.5 dB, SNR2 ≈ 306.2 dB)`

**Test case 3:** `rain.wav`, `window = 'hann'`, `M = 1024`, `N = 2048`, `H = 128`
→ expected: `(SNR1 ≈ 74.6 dB, SNR2 ≈ 304.3 dB)`

> Note: SNR values may vary slightly across machines due to floating-point differences.


In [9]:
# E4 - 2.2: Call the function compute_snr() for the 3 test cases mentioned, explain the results

### Your code here


**E4 - 2.3: Conceptual questions (answer here)**

1. For each test case, SNR2 is significantly higher than SNR1. What do the first and last `M` samples represent in the STFT reconstruction, and why are they noisier than the interior?
2. Compare SNR1 values across the three test cases (piano, sax, rain). The values differ substantially. What properties of the signal or the analysis parameters could explain these differences?
3. If you increase the hop size `H` while keeping `M` and `N` fixed, would you expect the SNR to increase or decrease? Why? Verify with one example.


## Part 3 – Computing band-wise energy envelopes

The function `compute_eng_env()` should compute energy envelopes for two frequency bands of an audio signal using the STFT.

**Frequency bands:**
- **Low band:** $0 < f < 3000$ Hz (excluding boundaries)
- **High band:** $3000 < f < 10000$ Hz (excluding boundaries)

**Steps:**
1. Load the audio file and run `stftAnal()` to get the magnitude spectra in dB for all frames.
2. Convert magnitude spectra from dB back to linear scale: $|X| = 10^{m_X/20}$.
3. For each frame, use `np.where()` to find bin indices within each band. The bin frequency is $f_k = k \cdot f_s / N$.
4. Compute per-frame band energy as the sum of squared linear magnitudes across the band's bins.
5. Convert each energy value to dB: $E_{\mathrm{dB}} = 10\log_{10}(E)$.

**Inputs:**
- `input_file` (str): path to a mono `.wav` file (44100 Hz)
- `window` (str): analysis window type
- `M` (int): window length (odd, power-of-2 convenient)
- `N` (int): FFT size (power of 2, $N > M$)
- `H` (int): hop size

**Output:** 2D np.array with shape `(num_frames, 2)` — column 0 is the low-band envelope, column 1 is the high-band envelope, both in dB.


In [10]:
# E4 - 3.1: Complete function compute_eng_env()

def compute_eng_env(input_file, window, M, N, H):
    """Compute band-wise energy envelopes of a given audio signal using the STFT.

    Args:
        input_file (string): input sound file (monophonic with sampling rate of 44100)
        window (string): analysis window type (choice of rectangular, triangular, hanning,
                hamming, blackman, blackmanharris)
        M (integer): analysis window size (odd positive integer)
        N (integer): FFT size (power of 2, such that N > M)
        H (integer): hop size for the stft computation

    Result:
        np.array: magnitude spectra of sound (2D array)
        np.array: 2D numpy array with energy envelope of band 0 < f < 3000 Hz (in dB) in first column, [:,0]
        np.array: energy envelope of band 3000 < f < 10000 Hz (in dB) in second column [:,1]
    """

    ### your code here


Test cases for `compute_eng_env()`:

**Test case 1:** `piano.wav`, `window = 'blackman'`, `M = 513`, `N = 1024`, `H = 128`
→ low band: bins 1–69 (69 bins); high band: bins 70–232 (163 bins)

**Test case 2:** `piano.wav`, `window = 'blackman'`, `M = 2047`, `N = 4096`, `H = 128`
→ low band: bins 1–278 (278 bins); high band: bins 279–928 (650 bins)

**Test case 3:** `sax-phrase-short.wav`, `window = 'hamming'`, `M = 513`, `N = 2048`, `H = 256`
→ low band: bins 1–139 (139 bins); high band: bins 140–464 (325 bins)

Plot both energy envelopes together with the signal's spectrogram (use `plt.pcolormesh` or reuse the code from `lectures/4-STFT/plots-code/spectrogram.py`). Use the same time-axis range for both. You should see sharp attacks and decays for the piano notes in test case 1 — compare with test case 2 (larger window) to observe the effect of window length on temporal resolution.


In [ ]:
# E4 - 3.2: Call compute_eng_env() for the 3 test cases and plot both energy envelopes
#            together with the spectrogram. Use the same time axis for all subplots.

### Your code here


**E4 - 3.3: Conceptual questions (answer here)**

1. From your spectrogram plot for test case 1 (`piano.wav`, `M=513`), visually identify a piano note onset and describe how the low-band and high-band envelopes behave at that moment. Which band reacts more sharply?
2. Compare envelopes from test case 1 (`M=513`) and test case 2 (`M=2047`) for the same file. How does increasing the window size affect the temporal sharpness of the note attacks in the envelope? Explain the underlying tradeoff.
3. Why is the DC bin (bin 0) excluded from the energy calculation? What kind of signal component does it represent, and what would happen to the energy values if you included it?


## Part 4 – Computing an onset detection function

The function `compute_odf()` should compute a half-wave-rectified energy-based ODF for two frequency bands, using the same low/high band definitions as Part 3.

**Steps:**
1. Compute the band-wise energy envelopes (in dB) exactly as in Part 3.
2. Compute the frame-to-frame energy difference: $O(l) = E(l) - E(l-1)$ for $l \geq 1$, with $O(0) = 0$.
3. Apply half-wave rectification: set all negative values to 0.

**Inputs:**
- `input_file` (str): path to a mono `.wav` file (44100 Hz)
- `window` (str): analysis window type
- `M` (int): window length (odd)
- `N` (int): FFT size (power of 2, $N > M$)
- `H` (int): hop size

**Output:** 2D np.array with shape `(num_frames, 2)` — column 0 is the ODF for the low band, column 1 for the high band.


In [12]:
# E4 4.1: Complete function compute_odf()

def compute_odf(input_file, window, M, N, H):
    """Compute a simple onset detection function (ODF) using the STFT.

    Args:
        input_file (str): input sound file (monophonic with sampling rate of 44100)
        window (str): analysis window type (rectangular, triangular, hanning, hamming, blackman, or blackmanharris)
        M (int): analysis window size (odd integer value)
        N (int): fft size (power of two, bigger or equal than than M)
        H (int): hop size for the STFT computation

    Result:
            np.array: magnitude spectra of sound (2D array)
            np.array: D numpy array with ODF computed in band 0 < f < 3000 Hz (in dB) in first column, [:,0]
            np.array: ODF computed of band 3000 < f < 10000 Hz (in dB) in second column [:,1]
    """

    ### your code here


Test cases for `compute_odf()` (same parameters as Part 3):

**Test case 1:** `piano.wav`, `window = 'blackman'`, `M = 513`, `N = 1024`, `H = 128`

**Test case 2:** `piano.wav`, `window = 'blackman'`, `M = 2047`, `N = 4096`, `H = 128`

**Test case 3:** `sax-phrase-short.wav`, `window = 'hamming'`, `M = 513`, `N = 2048`, `H = 256`

Plot each ODF together with the spectrogram of the corresponding signal (same time axis). For test case 1, you should see **5 clear peaks above 10 dB** in the high-band ODF, corresponding to the onsets of the piano notes.


In [ ]:
# E4 - 4.2: Call compute_odf() for the 3 test cases and plot the ODF functions together with
#            the spectrogram of the signal. Use the same time axis for both.

### Your code here


**E4 - 4.3: Conceptual questions (answer here)**

1. For test case 1 (`piano.wav`, `M=513`), count the peaks above 10 dB in the high-band ODF. Do they correspond to audible note onsets? Compare the peak locations in the low-band vs. high-band ODFs. Which band is more reliable for detecting piano note onsets, and why?
2. Compare the ODFs from test case 1 (`M=513`) and test case 2 (`M=2047`). How does window size affect the sharpness and height of the ODF peaks? Which would be easier to threshold for onset detection?
3. What is the effect of half-wave rectification on the ODF? Sketch (or plot) the ODF before and after rectification for one test case. What information is discarded, and why is it not needed for onset detection?
4. Could a single fixed dB threshold reliably detect onsets in all three test cases? Examine the ODF values across cases. If not, what alternative strategy would you propose?
